# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and utilizes AI-Ready standards for data infrastructure.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset published: {metadata.datePublished}")

## 2. Data Overview
Review the available record sets, fields, and their IDs.

We'll inspect what record sets, fields, and columns are present in the dataset using their `@id` identifiers. This ensures clarity and reproducibility in referencing dataset elements.

In [ ]:
# List available record sets and fields by their @id
record_sets = dataset.record_sets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"  @id: {rs['@id']} - name: {rs.get('name', '[no name]')}")

fields_by_rs = {}
for rs in record_sets:
    fields = dataset.fields(record_set=rs['@id'])
    fields_by_rs[rs['@id']] = fields

print("\nFields for each Record Set:")
for rs_id, fields in fields_by_rs.items():
    print(f"Record Set: {rs_id}")
    for fld in fields:
        print(f"    Field @id: {fld['@id']} - {fld.get('name', '[no name]')} (type: {fld.get('dataType', '')})")

### Example Record Preview
We'll display a preview of records from the primary record set using its `@id`.

In [ ]:
# Show example records from a record set
# Choose the primary record set (update this if another is more appropriate)
primary_record_set_id = record_sets[0]['@id'] if record_sets else None
if primary_record_set_id:
    for i, record in enumerate(dataset.records(record_set=primary_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from all record sets into DataFrames for further analysis.

Use the record set and field `@id`s from the overview section to ensure all references are consistent.

In [ ]:
# Extract data from all record sets by their @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print column overview for primary record set
if primary_record_set_id and primary_record_set_id in dataframes:
    print(f"Columns in Record Set {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping by key fields.

We will select a numeric field and a categorical field by their `@id` for demonstration.

In [ ]:
# Identify numeric and group fields by @id
# For this dataset, likely numeric: GAD-7 score, PHQ-9 score, PSQ score, e.g., use 'gad_7_score', 'phq_9_score', 'psq_score'
# Example field IDs (update if fields differ):
# Assume GAD-7 score field @id is 'https://api.app.sen.science/frontiers/7160411/gad7_score'
# Assume group field @id is 'https://api.app.sen.science/frontiers/7160411/level_of_education'
numeric_field_id = None
group_field_id = None
# Search for likely candidate fields
for fld in fields_by_rs.get(primary_record_set_id, []):
    fname = fld.get('name','').lower()
    if 'gad' in fname or 'phq' in fname or 'psq' in fname:
        numeric_field_id = fld['@id']
    if 'education' in fname:
        group_field_id = fld['@id']

# Use DataFrame
df = dataframes[primary_record_set_id]

# Check if numeric and group fields are present
if numeric_field_id in df.columns:
    print(f"Analyzing numeric field: {numeric_field_id}")
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the categorical field if present
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for analysis. Please check field IDs.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we'll show a histogram and a boxplot for the selected numeric field, and optionally a grouped barplot by education level.

In [ ]:
# Visualization examples
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    # Grouped barplot
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        group_means.sort_values().plot(kind='bar', figsize=(8,4))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have successfully loaded and explored the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

Key steps included:
- Loading metadata and records using the Croissant schema
- Inspecting available record sets and fields via their `@id`
- Extracting data into DataFrames
- Filtering and normalizing numeric fields, grouping by categorical fields
- Visualizing distributions and relationships

This workflow establishes a reproducible pattern for exploring complex FAIR datasets powered by Croissant standards, making them AI-ready for downstream purposes such as public health analysis and community research.

---
For further exploration, consider mapping additional fields, applying advanced statistical analysis, or linking this dataset with external data sources using their `@id` references.